# Session 3 · pandas: the grammar of data
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/urosgodnov/neoma-python-2627/blob/main/notebooks/S03_lecture_pandas.ipynb)  ·  *Python Programming for Business Analytics · NEOMA 2026/27 · dr. Uroš Godnov*

**Live on Zoom · 3 hours.** Filter, select, create, summarise, join, reshape, and handle missing values on purpose. Same data as Session 3 in R: **336,776 flights from New York in 2013** and the penguins.

| Part | Time | Topic |
|---|---|---|
| A1 | 0:00 | Method chaining: pandas' pipe |
| A2 | 0:10 | Rows: filter and sort |
| A3 | 0:35 | Columns: select, create, recode |
| A4 | 1:00 | Groups: summarise and transform |
| ☕ | **1:30** | **Break · 10 minutes** |
| B1 | 1:40 | Joins |
| B2 | 2:05 | Reshaping: wide ↔ long |
| B3 | 2:20 | Missing values, deliberately |
| B4 | 2:35 | Mini case: the route profitability desk |
| B5 | 2:55 | Wrap-up |

In [ ]:
# ⚙️ Setup: run this cell first (works in Google Colab and in Positron)
from pathlib import Path
import numpy as np
import pandas as pd

REPO = "https://raw.githubusercontent.com/urosgodnov/neoma-python-2627/main/"


def data_path(name):
    """Use the local course folder if you have it, otherwise read the file from GitHub."""
    for base in (Path("data"), Path("../data")):
        if (base / name).exists():
            return base / name
    return REPO + "data/" + name


pd.set_option("display.max_columns", 30)
print("Python ready · pandas", pd.__version__)
# The five tables of the R package nycflights13 (same data), stored as compressed CSV files.
flights, airlines, airports, planes, weather = (
    pd.read_csv(data_path(f"nycflights13/{name}.csv.gz"))
    for name in ["flights", "airlines", "airports", "planes", "weather"]
)

penguins = pd.read_csv(data_path("penguins.csv"))
flights.shape, airlines.shape, airports.shape, planes.shape, weather.shape

In [ ]:
flights.head()

---
## A1 · Method chaining: pandas' pipe

Wrap the chain in **parentheses** and put one step per line. Each method returns a new DataFrame, like each step after `|>`.

```r
flights |>                                   # R · dplyr
  filter(dest == "MIA") |>
  summarise(mean_delay = mean(arr_delay, na.rm = TRUE), .by = carrier) |>
  arrange(desc(mean_delay))
```

In [ ]:
(
    flights                                                  # Python · pandas
    .query("dest == 'MIA'")
    .groupby("carrier", as_index=False)
    .agg(mean_delay=("arr_delay", "mean"))
    .sort_values("mean_delay", ascending=False)
)

**The Rosetta stone** (keep it open all session)

| dplyr / tidyr | pandas |
|---|---|
| `filter()` | `df[mask]`, `.query()` |
| `filter_out()` | `df[~mask]` |
| `select()` | `df[["a", "b"]]`, `.filter(like=...)`, `.drop(columns=...)` |
| `mutate()` | `.assign()` |
| `rename()` | `.rename(columns={...})` |
| `arrange()` | `.sort_values()` |
| `summarise(.by = g)` | `.groupby(g).agg(new=("col", "func"))` |
| `mutate(.by = g)` | `.groupby(g)["col"].transform(func)` |
| `count()` | `.value_counts()`, `.groupby().size()` |
| `left_join()` | `.merge(how="left")` |
| `anti_join()` | `.merge(indicator=True)` + filter `"left_only"` |
| `pivot_wider()` / `pivot_longer()` | `.pivot_table()` / `.melt()` |
| `case_when()` | `np.select()`, `pd.cut()`, `Series.case_when()` |
| `recode_values()` | `.map(dict)` |
| `replace_values()` | `.replace(dict)` |

---
## A2 · Rows: filter and sort

A **boolean mask** is a column of `True/False`. `&` = and, `|` = or, `~` = not. Always put each condition in **parentheses**.

In [ ]:
late_to_florida = flights[
    (flights["month"] == 7)
    & (flights["origin"] == "JFK")
    & (flights["dest"].isin(["MIA", "FLL"]))
    & (flights["arr_delay"] > 60)
]
len(late_to_florida)

`.query()` reads like a sentence and is easy to chain. `@` refers to a Python variable.

In [ ]:
florida = ["MIA", "FLL"]
flights.query("month == 7 and origin == 'JFK' and dest in @florida and arr_delay > 60").shape

### ⚠️ Missing values and "not"
🔮 **Predict:** a comparison with a missing value (`NaN > 60`) returns `False` in pandas (in R it returns `NA`). So what does `~(arr_delay > 60)` do with cancelled flights?

In [ ]:
on_time_or_unknown = flights[~(flights["arr_delay"] > 60)]
on_time_known = flights[flights["arr_delay"] <= 60]
len(on_time_or_unknown), len(on_time_known), flights["arr_delay"].isna().sum()

💬 **Discuss:** `df[~mask]` behaves like dplyr 1.2's `filter_out()`: rows with missing values **stay**. `filter(!(...))` in R drops them. Which one do you want for a "remove the delayed flights" report?

In [ ]:
(
    flights
    .query("arr_delay > 180 or dep_time > 2300")
    .value_counts("origin")
)

In [ ]:
flights.nlargest(5, "dep_delay")[["month", "day", "carrier", "flight", "origin", "dest", "dep_delay"]]

🧪 **Try it (3 min):** how many flights to Boston (`BOS`) left more than 2 hours late in December?

In [ ]:
# your code here

---
## A3 · Columns: select, create, recode

In [ ]:
flights[["carrier", "distance", "air_time"]].head(3)

In [ ]:
flights.filter(like="delay").head(3)          # all columns whose name contains "delay"

`.assign()` creates columns inside a chain. `lambda d:` means *"the DataFrame at this step of the chain"* (like `.` in R).

In [ ]:
speeds = (
    flights
    .assign(speed_kmh=lambda d: d["distance"] * 1.609 / (d["air_time"] / 60))
    [["carrier", "distance", "air_time", "speed_kmh"]]
)
speeds.groupby("carrier")["speed_kmh"].median().sort_values(ascending=False).head().round(0)

### Recoding with a safety net
`.map(dict)` returns `NaN` for any value not in the dictionary. Check that nothing slipped through: that is `recode_values(unmatched = "error")`.

In [ ]:
AIRPORT_NAMES = {"JFK": "John F. Kennedy", "LGA": "LaGuardia", "EWR": "Newark Liberty"}


def recode_strict(s, mapping):
    out = s.map(mapping)
    unmatched = s[out.isna() & s.notna()].unique()
    if len(unmatched):
        raise ValueError(f"No mapping for: {list(unmatched)}")
    return out


flights.assign(origin_name=lambda d: recode_strict(d["origin"], AIRPORT_NAMES))["origin_name"].value_counts()

### Messy categories → ordered categories

In [ ]:
survey = pd.Series(["Strongly agree", "agree", "AGREE", "neutral", "Disagree", "n/a", None])
levels = ["strongly disagree", "disagree", "neutral", "agree", "strongly agree"]

clean = (
    survey
    .str.lower()
    .replace({"n/a": np.nan})
    .astype(pd.CategoricalDtype(levels, ordered=True))
)
clean

In [ ]:
clean.value_counts(sort=False), (clean >= "agree").sum()

### `case_when` in pandas
Ranges → `pd.cut()`. Any logic → `np.select()` (conditions are checked in order, first match wins).

In [ ]:
conditions = [
    flights["arr_delay"].isna(),
    flights["arr_delay"] <= 0,
    flights["arr_delay"] <= 30,
]
choices = ["cancelled", "on time", "small delay"]
flights["status"] = np.select(conditions, choices, default="big delay")
flights["status"].value_counts(normalize=True).round(3)

🧪 **Try it (3 min):** add a column `season` (`winter` for Dec–Feb, `spring`, `summer`, `autumn`) with `.map()` on `month`.

In [ ]:
# your code here

---
## A4 · Groups: summarise and transform

**Named aggregation:** `new_name=("column", "function")`. `as_index=False` gives a normal DataFrame back (like `.by` in R, no ungrouping needed).

In [ ]:
carrier_stats = (
    flights
    .groupby("carrier", as_index=False)
    .agg(
        n_flights=("flight", "size"),
        cancel_share=("arr_delay", lambda x: x.isna().mean()),
        mean_delay=("arr_delay", "mean"),
        p95_delay=("arr_delay", lambda x: x.quantile(0.95)),
    )
    .sort_values("cancel_share", ascending=False)
    .round(3)
)
carrier_stats

`.transform()` returns one value **per row** (the group statistic repeated): perfect for "compared with its group". 🆚 `mutate(.by = species)`.

In [ ]:
penguins["mass_z"] = penguins.groupby("species")["body_mass"].transform(lambda x: (x - x.mean()) / x.std())
penguins.loc[penguins["mass_z"].abs().nlargest(3).index, ["species", "sex", "body_mass", "mass_z"]].round(2)

💬 **Discuss:** why compare a penguin with its own species and not with all penguins? Find a business example of the same idea.

---
## ☕ Break · 10 minutes
---

## B1 · Joins

`validate="many_to_one"` is pandas' `relationship = "many-to-one"`: the join **fails** if the lookup table has duplicate keys, instead of silently multiplying your rows.

In [ ]:
airlines.head(3)

In [ ]:
flights_named = (
    flights
    .merge(airlines.rename(columns={"name": "airline"}), on="carrier", how="left", validate="many_to_one")
    .merge(
        airports[["faa", "name"]].rename(columns={"faa": "dest", "name": "dest_name"}),
        on="dest", how="left", validate="many_to_one",
    )
)
flights_named[["carrier", "airline", "dest", "dest_name"]].sample(5, random_state=3)

In [ ]:
flights_named["dest_name"].isna().sum(), flights_named.loc[flights_named["dest_name"].isna(), "dest"].unique()

### Anti-join: who is missing?
`indicator=True` adds a column `_merge` that tells you where each row was found.

In [ ]:
check = flights[["tailnum"]].dropna().drop_duplicates().merge(planes[["tailnum"]], how="left", indicator=True)
check["_merge"].value_counts()

### Joining on two keys: flights + weather

In [ ]:
flights_weather = flights.merge(
    weather[["origin", "time_hour", "visib", "wind_speed", "precip"]],
    on=["origin", "time_hour"], how="left", validate="many_to_one",
)
(
    flights_weather
    .assign(low_visibility=lambda d: d["visib"] < 2)
    .groupby("low_visibility")["dep_delay"].agg(["mean", "median", "size"])
    .round(1)
)

🔮 **Predict** before reading the result: is the **median** delay also higher in low visibility? What does the difference between mean and median tell you?

---
## B2 · Reshaping: wide ↔ long

Long data is for computers, wide data is for people.

In [ ]:
wide = flights.pivot_table(index="month", columns="origin", values="flight", aggfunc="size")
wide

In [ ]:
long = wide.reset_index().melt(id_vars="month", var_name="origin", value_name="n_flights")
long.head()

In [ ]:
pd.crosstab(flights["origin"], flights["status"], normalize="index").round(3)

---
## B3 · Missing values, deliberately

Every missing value needs a **decision**: drop, replace, impute by group, or flag. Write the decision down.

In [ ]:
penguins.isna().sum()

Impute missing body mass with the **median of the same species and sex**, and keep a flag so nobody mistakes an estimate for a measurement.

In [ ]:
by_species_sex = penguins.groupby(["species", "sex"])["body_mass"].transform("median")
step1 = penguins["body_mass"].fillna(by_species_sex)
penguins["body_mass"].isna().sum(), step1.isna().sum()

🔮 **Surprise:** nothing was imputed. 💬 **Discuss:** why? Look at the rows:

In [ ]:
penguins.loc[penguins["body_mass"].isna(), ["species", "island", "sex", "body_mass"]]

Their `sex` is missing too, so the group median does not exist. A **fallback** to the species median solves it, and the flag records what we did.

In [ ]:
by_species = penguins.groupby("species")["body_mass"].transform("median")
penguins = penguins.assign(
    mass_imputed=penguins["body_mass"].isna(),
    body_mass_filled=penguins["body_mass"].fillna(by_species_sex).fillna(by_species),
)
penguins["mass_imputed"].sum(), penguins["body_mass_filled"].isna().sum()

💬 **Discuss:** these two penguins have **no measurements at all**. Is imputing them honest, or should we drop them? Write your decision in one sentence.

---
## B4 · Mini case: the route profitability desk

The network planning team asks: *"Which routes with at least 1,000 flights have the highest share of flights delayed by more than 30 minutes?"*

In [ ]:
routes = (
    flights
    .groupby(["origin", "dest"], as_index=False)
    .agg(
        n_flights=("flight", "size"),
        delayed_share=("arr_delay", lambda x: (x > 30).mean()),
        distance=("distance", "first"),
    )
    .query("n_flights >= 1000")
    .merge(airports[["faa", "name"]].rename(columns={"faa": "dest", "name": "dest_name"}), on="dest", how="left")
    .assign(
        origin_name=lambda d: d["origin"].map(AIRPORT_NAMES),
        haul=lambda d: pd.cut(d["distance"], [0, 500, 1500, np.inf], labels=["Short", "Medium", "Long"], right=False),
    )
    .sort_values("delayed_share", ascending=False)
)
top10 = routes.head(10)[["origin_name", "dest_name", "haul", "n_flights", "delayed_share"]]
top10.style.format({"delayed_share": "{:.1%}", "n_flights": "{:,}"})

In [ ]:
worst = top10.iloc[0]
print(
    f"The worst route is {worst['origin_name']} → {worst['dest_name']}: "
    f"{worst['delayed_share']:.1%} of its {worst['n_flights']:,} flights arrived more than 30 minutes late."
)

💬 **Discuss:** is `delayed_share` computed on all flights or only on flights that arrived? Does it matter here?

---
## B5 · Wrap-up

- chains in parentheses = pandas' pipe
- masks with `&`, `|`, `~` and parentheses, or `.query()`
- `.assign()`, `.map()` with a safety net, ordered categories, `np.select()`
- `.groupby().agg()` with named aggregation, `.transform()` for per-row group statistics
- `.merge(validate=...)`, `indicator=True` for anti-joins
- `.pivot_table()` / `.melt()`, and a written decision for every missing value

**Before Session 4:** practice notebook **S03_practice** and the **Session 3 quiz**.

📺 Optional: Keith Galli's *Complete Python Pandas Data Science Tutorial (2025)* in Moodle.